# 07 - Evaluasi Kuantitatif & Komparasi Model
### Proyek CITRA SERUPA — Sistem Rekomendasi Gambar Fashion Berdasarkan Kemiripan Visual

---

**Tujuan Notebook:**
Notebook ini menyajikan pengujian ilmiah terukur (*Quantitative Evaluation*) sesuai kaidah penelitian *Information Retrieval*:
1. **Metrik Evaluasi Ilmiah:**
   - **Recall@K (Top-K Hit Rate / Accuracy):** Mengukur seberapa sering produk yang tepat berhasil ditemukan dalam Top-1, Top-5, Top-10, dan Top-20 rekomendasi.
   - **mAP@10 (Mean Average Precision at K=10):** Mengukur ketepatan peringkat dan relevansi seluruh rekomendasi hingga posisi ke-10.
   - **MRR (Mean Reciprocal Rank):** Mengukur seberapa cepat (posisi ke berapa) rekomendasi benar pertama kali muncul.
2. **Eksperimen Model Pembanding (Benchmarking):**
   - Mengekstrak fitur menggunakan model alternatif ringan: **MobileNetV3-Large** (960 Dimensi).
   - Membandingkan performa *Head-to-Head* antara **ResNet-50 (Baseline)** vs **MobileNetV3-Large (Pembanding)** dari segi akurasi, kecepatan inferensi, dan ukuran fitur.
3. **Visualisasi Kurva & Tabel Ilmiah:** Menyediakan grafik perbandingan dan tabel ringkasan yang siap disalin ke bab Hasil & Pembahasan laporan kuliah/skripsi.

## 1. Hubungkan Google Colab ke Google Drive & Muat Data ResNet-50

In [ ]:
from google.colab import drive
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Deteksi lokasi folder proyek
base_dir = '/content/drive/MyDrive/CITRA-SERUPA'
if not os.path.exists(base_dir):
    base_dir = '/content/drive/MyDrive/CITRA_SERUPA'

dataset_dir = None
images_dir = None
for d in ['dataset', 'dataset ']:
    cand = os.path.join(base_dir, d, 'images')
    if os.path.exists(cand) and len(os.listdir(cand)) > 0:
        dataset_dir = os.path.join(base_dir, d)
        images_dir = cand
        break

features_dir = os.path.join(base_dir, 'features')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 3. Muat Fitur ResNet-50 yang Telah Disimpan
print("⏳ Memuat fitur ResNet-50...")
resnet_emb = np.load(os.path.join(features_dir, 'embeddings.npy'))
image_names = np.load(os.path.join(features_dir, 'image_names.npy'))
item_ids = np.load(os.path.join(features_dir, 'item_ids.npy'))

print("=" * 65)
print("✅ DATA EVALUASI SIAP:")
print(f" - Total Citra Diuji       : {len(image_names):,} gambar")
print(f" - Dimensi Vektor ResNet-50: {resnet_emb.shape} (2048 Dimensi)")
print(f" - Total Produk Unik       : {len(np.unique(item_ids)):,} item pakaian")
print(f" - Akselerasi Komputasi    : {device}")
print("=" * 65)

## 2. Definisi Algoritma Perhitungan Metrik Evaluasi (Recall@K, mAP@10, MRR)

In [ ]:
def calculate_retrieval_metrics(embeddings, item_ids, k_values=[1, 5, 10, 20]):
    """
    Menghitung Recall@K, mAP@10, dan MRR secara menyeluruh untuk seluruh citra dalam galeri
    menggunakan protokol Leave-One-Out (kueri diuji terhadap seluruh galeri kecuali dirinya sendiri).
    """
    num_items = len(embeddings)
    print(f"🔄 Menghitung matriks kemiripan untuk {num_items:,} x {num_items:,} pasangan citra...")
    
    # Karena embeddings sudah L2-normalized, Cosine Sim = Dot Product
    # Matriks sim berukuran (N, N) memakan memori ~52 MB (sangat ringan)
    sim_matrix = np.dot(embeddings, embeddings.T)
    
    # Hilangkan kemiripan terhadap diri sendiri dengan memberi nilai -inf di diagonal
    np.fill_diagonal(sim_matrix, -np.inf)
    
    # Urutkan indeks galeri berdasarkan skor kemiripan tertinggi (descending)
    max_k = max(k_values)
    top_ranks = np.argsort(sim_matrix, axis=1)[:, ::-1][:, :max_k]
    
    # Inisialisasi penyimpan skor
    recalls = {k: [] for k in k_values}
    average_precisions_10 = []
    reciprocal_ranks = []
    
    print("📊 Mengalkulasi metrik Recall@K, mAP@10, dan MRR...")
    for i in range(num_items):
        query_item = item_ids[i]
        ranked_items = item_ids[top_ranks[i]]
        
        # Hitung total gambar yang relevan (baju yang sama) di galeri
        total_relevant_in_gallery = np.sum(item_ids == query_item) - 1
        if total_relevant_in_gallery <= 0:
            continue  # Abaikan jika baju tidak memiliki pasangan sama sekali
            
        # Boolean array: True jika produk sama (Match Ground Truth)
        matches = (ranked_items == query_item)
        
        # 1. Recall@K (Apakah ada setidaknya 1 gambar relevan dalam Top-K?)
        for k in k_values:
            has_hit = np.any(matches[:k])
            recalls[k].append(1.0 if has_hit else 0.0)
            
        # 2. Average Precision @ 10 (AP@10)
        hits_at_k = 0
        precisions_sum = 0.0
        for rank_idx in range(min(10, max_k)):
            if matches[rank_idx]:
                hits_at_k += 1
                precisions_sum += hits_at_k / (rank_idx + 1)
        denominator = min(total_relevant_in_gallery, 10)
        ap_10 = precisions_sum / denominator if denominator > 0 else 0.0
        average_precisions_10.append(ap_10)
        
        # 3. Reciprocal Rank (MRR)
        first_hit_indices = np.where(matches)[0]
        if len(first_hit_indices) > 0:
            reciprocal_ranks.append(1.0 / (first_hit_indices[0] + 1))
        else:
            reciprocal_ranks.append(0.0)
            
    # Rata-ratakan seluruh metrik
    results = {
        'Recall@1': np.mean(recalls[1]) * 100,
        'Recall@5': np.mean(recalls[5]) * 100,
        'Recall@10': np.mean(recalls[10]) * 100,
        'Recall@20': np.mean(recalls[20]) * 100,
        'mAP@10': np.mean(average_precisions_10) * 100,
        'MRR': np.mean(reciprocal_ranks) * 100,
        'Evaluated_Queries': len(average_precisions_10)
    }
    
    return results

print("✅ Fungsi evaluasi kuantitatif ilmiah siap digunakan!")

## 3. Eksekusi Evaluasi Model Baseline (ResNet-50)

In [ ]:
import time

start_time = time.time()
resnet_results = calculate_retrieval_metrics(resnet_emb, item_ids)
eval_duration = time.time() - start_time

print("=" * 65)
print("🏆 HASIL EVALUASI KUANTITATIF — MODEL BASELINE RESNET-50")
print("=" * 65)
print(f"Total Kueri Terevaluasi  : {resnet_results['Evaluated_Queries']:,} citra")
print(f"Recall@1  (Top-1 Match)  : {resnet_results['Recall@1']:.2f}%")
print(f"Recall@5  (Top-5 Match)  : {resnet_results['Recall@5']:.2f}%")
print(f"Recall@10 (Top-10 Match) : {resnet_results['Recall@10']:.2f}%")
print(f"Recall@20 (Top-20 Match) : {resnet_results['Recall@20']:.2f}%")
print(f"mAP@10    (Mean Avg Prec): {resnet_results['mAP@10']:.2f}%")
print(f"MRR       (Mean Recip Rk): {resnet_results['MRR']:.2f}%")
print(f"Waktu Evaluasi Total     : {eval_duration:.2f} detik")
print("=" * 65)

## 4. Ekstraksi Fitur Model Pembanding (MobileNetV3-Large)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

print("⏳ Mempersiapkan Model Pembanding: MobileNetV3-Large (Lightweight Architecture)...")

# 1. Muat MobileNetV3-Large Pretrained ImageNet
mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
# Ganti classifier head dengan Identity agar menghasilkan vektor fitur 960 dimensi
mobilenet.classifier = nn.Identity()
mobilenet = mobilenet.to(device)
mobilenet.eval()

# 2. Siapkan DataLoader
transform_pipe = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class SimpleDataset(Dataset):
    def __init__(self, names, img_dir, tfm):
        self.names = names
        self.img_dir = img_dir
        self.tfm = tfm
    def __len__(self):
        return len(self.names)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.img_dir, self.names[i]))
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return self.tfm(img)

loader = DataLoader(SimpleDataset(image_names, images_dir, transform_pipe), batch_size=64, shuffle=False, num_workers=2)

# 3. Ekstraksi Fitur MobileNetV3 pada GPU
mobilenet_features = []
start_ext = time.time()

with torch.no_grad():
    for batch in tqdm(loader, desc="Ekstraksi MobileNetV3-Large"):
        batch = batch.to(device)
        feats = mobilenet(batch)  # Dimensi: [batch, 960]
        feats = F.normalize(feats, p=2, dim=1)  # L2 Normalization
        mobilenet_features.append(feats.cpu().numpy())

mobilenet_emb = np.vstack(mobilenet_features)
ext_duration = time.time() - start_ext

# Simpan fitur ke Drive
mobilenet_save_path = os.path.join(features_dir, 'embeddings_mobilenet.npy')
np.save(mobilenet_save_path, mobilenet_emb)

print("=" * 65)
print("📱 EKSTRAKSI MOBILENETV3-LARGE SELESAI:")
print(f" - Dimensi Vektor       : {mobilenet_emb.shape} (960 Dimensi)")
print(f" - Waktu Ekstraksi Total: {ext_duration:.2f} detik")
print(f" - File Tersimpan di    : {mobilenet_save_path}")
print("=" * 65)

## 5. Eksekusi Evaluasi Model Pembanding (MobileNetV3-Large)

In [ ]:
print("⏳ Mengevaluasi metrik pencarian untuk MobileNetV3-Large...")
mobilenet_results = calculate_retrieval_metrics(mobilenet_emb, item_ids)

print("=" * 65)
print("📱 HASIL EVALUASI KUANTITATIF — MODEL MOBILENETV3-LARGE")
print("=" * 65)
print(f"Recall@1  (Top-1 Match)  : {mobilenet_results['Recall@1']:.2f}%")
print(f"Recall@5  (Top-5 Match)  : {mobilenet_results['Recall@5']:.2f}%")
print(f"Recall@10 (Top-10 Match) : {mobilenet_results['Recall@10']:.2f}%")
print(f"Recall@20 (Top-20 Match) : {resnet_results['Recall@20']:.2f}%")
print(f"mAP@10    (Mean Avg Prec): {mobilenet_results['mAP@10']:.2f}%")
print(f"MRR       (Mean Recip Rk): {mobilenet_results['MRR']:.2f}%")
print("=" * 65)

## 6. Tabel Perbandingan Head-to-Head & Grafik Visualisasi Ilmiah

In [ ]:
# 1. Buat Tabel Perbandingan DataFrame
comparison_data = {
    'Arsitektur Model': ['ResNet-50 (Baseline)', 'MobileNetV3-Large (Pembanding)'],
    'Dimensi Fitur': ['2.048 Dimensi', '960 Dimensi'],
    'Recall@1 (%)': [f"{resnet_results['Recall@1']:.2f}%", f"{mobilenet_results['Recall@1']:.2f}%"],
    'Recall@5 (%)': [f"{resnet_results['Recall@5']:.2f}%", f"{mobilenet_results['Recall@5']:.2f}%"],
    'Recall@10 (%)': [f"{resnet_results['Recall@10']:.2f}%", f"{mobilenet_results['Recall@10']:.2f}%"],
    'mAP@10 (%)': [f"{resnet_results['mAP@10']:.2f}%", f"{mobilenet_results['mAP@10']:.2f}%"],
    'MRR (%)': [f"{resnet_results['MRR']:.2f}%", f"{mobilenet_results['MRR']:.2f}%"],
    'Karakteristik Utama': ['Akurasi representasi tinggi', 'Sangat ringan & efisien']
}

df_comparison = pd.DataFrame(comparison_data)
print("=" * 85)
print("📊 TABEL PERBANDINGAN PERFORMA RESNET-50 VS MOBILENETV3-LARGE")
print("=" * 85)
display(df_comparison)
print("=" * 85)

# 2. Grafik Komparasi Recall@K dan mAP@10
metrics_names = ['Recall@1', 'Recall@5', 'Recall@10', 'Recall@20', 'mAP@10', 'MRR']
resnet_scores = [resnet_results[m] for m in metrics_names]
mobilenet_scores = [mobilenet_results[m] for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35

plt.figure(figsize=(12, 6))
bars1 = plt.bar(x - width/2, resnet_scores, width, label='ResNet-50 (2048-dim)', color='#1f77b4', edgecolor='black')
bars2 = plt.bar(x + width/2, mobilenet_scores, width, label='MobileNetV3-Large (960-dim)', color='#ff7f0e', edgecolor='black')

plt.title('Perbandingan Kinerja Rekomendasi Citra: ResNet-50 vs MobileNetV3-Large', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Metrik Evaluasi', fontsize=12, fontweight='bold')
plt.ylabel('Skor Persentase (%)', fontsize=12, fontweight='bold')
plt.xticks(x, metrics_names, fontsize=11)
plt.ylim(0, max(max(resnet_scores), max(mobilenet_scores)) + 15)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(fontsize=11)

# Tampilkan nilai angka di atas masing-masing batang
for bar in bars1:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1f77b4')

for bar in bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold', color='#ff7f0e')

plt.tight_layout()
plt.show()

## 7. Rangkuman Ilmiah Siap Pakai untuk Laporan Skripsi / Tugas Kuliah

In [ ]:
best_model = "ResNet-50" if resnet_results['mAP@10'] >= mobilenet_results['mAP@10'] else "MobileNetV3-Large"

print("=" * 75)
print("📝 DRAFT PEMBAHASAN HASIL EVALUASI UNTUK LAPORAN:")
print("=" * 75)
print(f"1. Model Terbaik: {best_model} unggul dalam menangkap fitur visual busana.")
print(f"2. Pada pengujian Top-1 (Recall@1), ResNet-50 mencapai {resnet_results['Recall@1']:.2f}% sedangkan MobileNetV3 {mobilenet_results['Recall@1']:.2f}%.")
print(f"3. Pada pengujian Top-10 (Recall@10), sistem mencapai {resnet_results['Recall@10']:.2f}% probabilitas menemukan produk yang sama.")
print(f"4. Nilai mAP@10 ({resnet_results['mAP@10']:.2f}%) dan MRR ({resnet_results['MRR']:.2f}%) membuktikan bahwa peringkat rekomendasi")
print("   yang dihasilkan model berada pada posisi urutan teratas yang sangat relevan secara visual.")
print("=" * 75)